## Win Probability Model — For CFS Component 1

### What is Win Probability Added (WPA)
WPA measures how much a batter's innings changed their team's
probability of winning. This is fundamentally different from
binary win/loss metrics.

Example:
- Before Dhoni's innings: India win probability = 25%
- After Dhoni's innings: India win probability = 80%
- WPA = +55% (extremely clutch)

### Why this replaces Component 1
Component 1 (win% when 50+) was binary — either team won or lost.
WPA is continuous — captures HOW MUCH the batter changed the game.
A batter scoring 80 in a comfortable chase contributes less WPA
than one scoring 80 when team needed 120 off 60 balls.

### Model Architecture
Logistic Regression trained on historical match situations.
Input features at each ball:
- innings (1 or 2)
- over_num (0-49)
- cumulative_wickets (0-10)
- cumulative_runs
- target (for innings 2)
- run_rate_required (for innings 2)
- wickets_remaining

Output: probability that batting team wins from this situation

### Training approach
Label each ball: did batting team eventually win this match?
Train on all 1.3M balls → learns win probability from any situation
Then apply to each batter's specific balls faced

In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pickle
import os

# load master data
master_df = pd.read_parquet(r"D:\CricMetric-AI\data\processed\master_odi.parquet")
print(f"Loaded: {master_df.shape}")

Loaded: (1349408, 33)


In [5]:
# load match results we need
# recompute match results from scratch
match_score = master_df.groupby(
    ['match_id', 'innings', 'batting_team']
)['cumulative_runs'].max().reset_index()
match_score.columns = ['match_id', 'innings', 'batting_team', 'team_match_total']

match_score_main = match_score[match_score['innings'].isin([1, 2])]

match_pivot = match_score_main.pivot(
    index='match_id', columns='innings', values='team_match_total'
).reset_index()
match_pivot.columns = ['match_id', 'innings1_total', 'innings2_total']


In [6]:
team_names = match_score_main.pivot(
    index='match_id', columns='innings', values='batting_team'
).reset_index()
team_names.columns = ['match_id', 'team_innings1', 'team_innings2']

match_results = match_pivot.merge(team_names, on='match_id')
match_results['winner'] = match_results.apply(
    lambda row: row['team_innings1'] if row['innings1_total'] > row['innings2_total']
    else row['team_innings2'] if row['innings2_total'] > row['innings1_total']
    else 'Tie', axis=1
)

print(f"Match results computed: {len(match_results)} matches")



Match results computed: 2544 matches


In [7]:
# innings 1 totals for target
innings1_totals = master_df[
    master_df['innings']==1
].groupby('match_id')['cumulative_runs'].max().reset_index()
innings1_totals.columns = ['match_id', 'innings1_total']
innings1_totals['target'] = innings1_totals['innings1_total'] + 1

print(f"Innings 1 totals: {len(innings1_totals)} matches")

Innings 1 totals: 2544 matches


In [8]:
# merge match winner into master_df
wp_data = master_df.copy()

wp_data = wp_data.merge(
    match_results[['match_id', 'winner']], 
    on='match_id', how='left'
)

# did batting team win?
wp_data['batting_team_won'] = (
    wp_data['batting_team'] == wp_data['winner']
).astype(int)

# merge target for innings 2
wp_data = wp_data.merge(
    innings1_totals[['match_id', 'target']], 
    on='match_id', how='left'
)



In [9]:
# build features
wp_data['balls_bowled'] = (
    wp_data['over_num'] * 6 + wp_data['ball_num']
).clip(1)

wp_data['balls_remaining'] = (300 - wp_data['balls_bowled']).clip(0)
wp_data['wickets_remaining'] = 10 - wp_data['cumulative_wickets']

wp_data['runs_remaining'] = wp_data.apply(
    lambda x: max(0, x['target'] - x['cumulative_runs'])
    if x['innings'] == 2 else 0, axis=1
)

wp_data['rrr'] = wp_data.apply(
    lambda x: x['runs_remaining'] / max(x['balls_remaining'], 1) * 6
    if x['innings'] == 2 else 0, axis=1
)

wp_data['current_rr'] = (
    wp_data['cumulative_runs'] / wp_data['balls_bowled'] * 6
)

print(f"Features built. Shape: {wp_data.shape}")
print(f"\nClass distribution:")
print(wp_data['batting_team_won'].value_counts())

Features built. Shape: (1349408, 42)

Class distribution:
batting_team_won
0    678414
1    670994
Name: count, dtype: int64


In [10]:
# select features for training
feature_cols = [
    'innings', 'over_num', 'cumulative_runs', 'cumulative_wickets',
    'wickets_remaining', 'balls_remaining', 'runs_remaining',
    'rrr', 'current_rr'
]

# remove rows with missing values
wp_clean = wp_data[feature_cols + ['batting_team_won']].dropna()
print(f"Clean training rows: {len(wp_clean)}")

X = wp_clean[feature_cols]
y = wp_clean['batting_team_won']

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

# scale features
scaler_wp = StandardScaler()
X_train_scaled = scaler_wp.fit_transform(X_train)
X_test_scaled = scaler_wp.transform(X_test)

# train logistic regression
wp_model = LogisticRegression(max_iter=1000, random_state=42)
wp_model.fit(X_train_scaled, y_train)

# evaluate
y_pred_proba = wp_model.predict_proba(X_test_scaled)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"\nModel AUC-ROC: {auc:.4f}")

Clean training rows: 1349408
Train size: 1079526
Test size: 269882

Model AUC-ROC: 0.8036


In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb
import time

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': CalibratedClassifierCV(
        DecisionTreeClassifier(max_depth=8, random_state=42), cv=3
    ),
    'Random Forest': CalibratedClassifierCV(
        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), cv=3
    ),
    'XGBoost': CalibratedClassifierCV(
        xgb.XGBClassifier(n_estimators=100, max_depth=6, 
                          random_state=42, eval_metric='logloss'), cv=3
    ),
    'Gradient Boosting': CalibratedClassifierCV(
        GradientBoostingClassifier(n_estimators=100, max_depth=4, 
                                   random_state=42), cv=3
    ),
}

results = {}
for name, model in models.items():
    print(f"Training {name}...")
    start = time.time()
    model.fit(X_train_scaled, y_train)
    train_time = time.time() - start
    
    y_pred = model.predict_proba(X_test_scaled)[:, 1]
    auc = roc_auc_score(y_test, y_pred)
    results[name] = {'AUC': auc, 'Time': f"{train_time:.1f}s"}
    print(f"  AUC: {auc:.4f} | Time: {train_time:.1f}s")

print("\n--- Final Comparison ---")
for name, metrics in sorted(results.items(), key=lambda x: x[1]['AUC'], reverse=True):
    print(f"{name:25s} AUC: {metrics['AUC']:.4f}  Time: {metrics['Time']}")

Training Logistic Regression...
  AUC: 0.8036 | Time: 2.7s
Training Decision Tree...
  AUC: 0.8046 | Time: 13.7s
Training Random Forest...
  AUC: 0.8445 | Time: 56.1s
Training XGBoost...
  AUC: 0.8198 | Time: 5.7s
Training Gradient Boosting...
  AUC: 0.8112 | Time: 714.0s

--- Final Comparison ---
Random Forest             AUC: 0.8445  Time: 56.1s
XGBoost                   AUC: 0.8198  Time: 5.7s
Gradient Boosting         AUC: 0.8112  Time: 714.0s
Decision Tree             AUC: 0.8046  Time: 13.7s
Logistic Regression       AUC: 0.8036  Time: 2.7s


In [12]:
print(X_train_scaled.dtype)

float64


In [13]:
X_train_scaled = X_train_scaled.astype(np.float32)
X_test_scaled  = X_test_scaled.astype(np.float32)

In [14]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
import numpy as np

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [8, 12, 16, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.3]
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)




In [15]:

# use 3-fold CV, try 20 random combinations
random_search = RandomizedSearchCV(
    rf_base,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
random_search.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {random_search.best_params_}")
print(f"Best CV AUC: {random_search.best_score_:.4f}")

# evaluate on test set
y_pred_tuned = random_search.best_estimator_.predict_proba(X_test_scaled)[:, 1]
auc_tuned = roc_auc_score(y_test, y_pred_tuned)
print(f"Test AUC (tuned): {auc_tuned:.4f}")
print(f"Previous AUC: 0.8445")
print(f"Improvement: +{(auc_tuned - 0.8445):.4f}")

Fitting 3 folds for each of 10 candidates, totalling 30 fits


d:\CricMetric-AI\cricmetric_env\Lib\site-packages\sklearn\model_selection\_validation.py:489: FitFailedWarning: 
4 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
joblib.externals.loky.process_executor._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "d:\CricMetric-AI\cricmetric_env\Lib\site-packages\joblib\_utils.py", line 109, in __call__
    return self.func(**kwargs)
           ~~~~~~~~~^^^^^^^^^^
  File "d:\CricMetric-AI\cricmetric_env\Lib\site-packages\joblib\parallel.py", line 607, in __call__
    return [func(*args, **kwargs) for func, args, kwargs in self.items]
            ~~~~^^^^^^^^^^^^^^^^^
  File "d:\CricMetric-AI\cricmetric_env\Lib\s


Best parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 16}
Best CV AUC: 0.8463
Test AUC (tuned): 0.8491
Previous AUC: 0.8445
Improvement: +0.0046


In [16]:
from sklearn.calibration import CalibratedClassifierCV

# calibrate tuned model
calibrated_rf = CalibratedClassifierCV(
    random_search.best_estimator_, 
    cv=3, 
    method='isotonic'
)
calibrated_rf.fit(X_train_scaled, y_train)

# check AUC after calibration
y_pred_calibrated = calibrated_rf.predict_proba(X_test_scaled)[:, 1]
auc_calibrated = roc_auc_score(y_test, y_pred_calibrated)
print(f"Calibrated tuned RF AUC: {auc_calibrated:.4f}")

Calibrated tuned RF AUC: 0.8465


In [17]:
import pandas as pd

test_scenarios = pd.DataFrame({
    'innings': [2, 2, 2, 2, 1],
    'over_num': [45, 45, 10, 48, 25],
    'cumulative_runs': [250, 150, 80, 295, 150],
    'cumulative_wickets': [2, 8, 3, 9, 3],
    'wickets_remaining': [8, 2, 7, 1, 7],
    'balls_remaining': [30, 30, 240, 12, 150],
    'runs_remaining': [50, 150, 220, 5, 0],
    'rrr': [10.0, 30.0, 5.5, 2.5, 0],
    'current_rr': [33.3, 20.0, 48.0, 36.6, 36.0]
})

scenarios_scaled = scaler_wp.transform(test_scenarios)
win_probs_final = calibrated_rf.predict_proba(scenarios_scaled)[:, 1]

descriptions = [
    "Chase: need 50 off 30 balls, 8 wickets left (comfortable)",
    "Chase: need 150 off 30 balls, 2 wickets left (near impossible)",
    "Chase: need 220 off 240 balls, 7 wickets left (early, manageable)",
    "Chase: need 5 off 12 balls, 1 wicket left (very tense)",
    "1st innings: 150/3 after 25 overs (decent position)"
]

print("Final Win Probability Sanity Check:")
for desc, prob in zip(descriptions, win_probs_final):
    print(f"\n{desc}")
    print(f"Win probability: {prob*100:.1f}%")

Final Win Probability Sanity Check:

Chase: need 50 off 30 balls, 8 wickets left (comfortable)
Win probability: 58.1%

Chase: need 150 off 30 balls, 2 wickets left (near impossible)
Win probability: 11.5%

Chase: need 220 off 240 balls, 7 wickets left (early, manageable)
Win probability: 57.5%

Chase: need 5 off 12 balls, 1 wicket left (very tense)
Win probability: 81.3%

1st innings: 150/3 after 25 overs (decent position)
Win probability: 48.2%


In [19]:
import pickle

with open(r"D:\CricMetric-AI\data\processed\win_prob_model.pkl", 'wb') as f:
    pickle.dump(calibrated_rf, f)

with open(r"D:\CricMetric-AI\data\processed\win_prob_scaler.pkl", 'wb') as f:
    pickle.dump(scaler_wp, f)

print("Final model saved:")
print(f"  Model: win_prob_model.pkl")
print(f"  Scaler: win_prob_scaler.pkl")
print(f"  AUC: 0.8465")
print(f"  Algorithm: Calibrated Tuned Random Forest")

Final model saved:
  Model: win_prob_model.pkl
  Scaler: win_prob_scaler.pkl
  AUC: 0.8465
  Algorithm: Calibrated Tuned Random Forest
